In [ ]:
# Load all the necessary libraries
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from statannotations.Annotator import Annotator

# Supress all warnings (bcoz they're quite annoying)
import warnings

warnings.filterwarnings("ignore")

import sys
sys.path.append("..")

In [ ]:
def get_class_palette():
    
    colors = sns.color_palette("colorblind")
    pal = {
        "young": colors[2],
        "aged": colors[3],
    }
    
    return pal

def get_microscopist_palette():
    
    colors = sns.color_palette("colorblind")
    pal = {
        "Microscopist 1": colors[1],
        "Microscopist 2": colors[9],
        "Microscopist 3": colors[4],
    }
    
    return pal

In [ ]:
nuc_df = pd.read_csv(
    "../results/pyradiomics/nobin_0.1_z_score_all_df.csv",
    index_col=0,
)

pal = get_class_palette()

In [ ]:
my_df = nuc_df
condition = my_df.condition

czi_metadata_filt = pd.read_csv("../results/czi_metadata_filt.csv")
my_df = my_df.merge(czi_metadata_filt, left_on="czi_path", right_on="path")
my_df["condition"] = condition
my_df = my_df[my_df["condition"].isin(["young", "aged"])]
my_df = my_df.reset_index(drop=True)

In [ ]:
# Filter those features that belong to the filter WAVELET H
my_df = my_df[[col for col in my_df.columns if not col.startswith("wavelet-H")]]

In [ ]:
my_df

In [ ]:
features = my_df.columns[:-36]
features

In [ ]:
my_df.describe().T

In [ ]:
# Drop columns that contain NaN values
# nuc_df = nuc_df.dropna()
sns.set(rc={"figure.figsize": (20, 16)})
sns.set_style("whitegrid")

int_df = my_df[features]

# Drop all the columns that have the same value for all rows
nunique = int_df.nunique()
cols_drop = nunique[nunique == 1].index
int_df = int_df.drop(cols_drop, axis=1)

In [ ]:
print(cols_drop)

In [ ]:
int_df

In [ ]:
num_feats = int_df.select_dtypes(include='number').columns

# Z-score normalization
norm_df = int_df.copy()
norm_df[num_feats] = (int_df[num_feats] - int_df[num_feats].mean()) / int_df[num_feats].std()

In [ ]:
corr_mat = norm_df.corr()
mask = np.abs(corr_mat) > 0.85

# Get the gene pairs that are highly correlated
highly_corr_pairs = np.where(mask)

In [ ]:
# Get the highly correlated gene pairs
corr_cols_to_remove = set()

for i, j in zip(*highly_corr_pairs):
    if i != j:
        col_to_remove = corr_mat.columns[j]
        if col_to_remove == "log-sigma-0-5-mm-3D_glcm_SumEntropy":
            col_to_remove = corr_mat.columns[i] 
        corr_cols_to_remove.add(col_to_remove)

# Filter the genes to keep
filtered_cols = [c for c in int_df.columns if c not in corr_cols_to_remove]
features = [c for c in features if c not in corr_cols_to_remove]

# Create a new AnnData object with the filtered genes
filtered_df = my_df[filtered_cols].copy()

# Check the new shape
print(f"Original shape: {my_df.shape}")
print(f"Filtered shape: {filtered_df.shape}")

In [ ]:
filtered_cols

In [ ]:
sns.set(rc={"figure.figsize": (25, 16)})
sns.set_style("whitegrid")

sns.clustermap(
    norm_df.corr(), dendrogram_ratio=0.1, yticklabels=False, xticklabels=False
)

In [ ]:
sns.clustermap(
    filtered_df.corr(), dendrogram_ratio=0.1, yticklabels=False, 
    xticklabels=False, vmin=-1, vmax=1,
)

In [ ]:
col_to_delete = ['npz_path', 'original_num_channels', 'original_sigma_noise',
       'original_intensity_sum', 'original_intensity_mean',
       'original_intensity_max', 'nuc_id', 'batch_id', 'czi_path', 'condition']

In [ ]:
out = filtered_df.copy()
out["condition"] = my_df["condition"]
out["czi_path"] = my_df["czi_path"]
out.to_csv("../results/pyradiomics/supp_table1_filtered_features.csv")

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# PCA is affected by scale, first we need to standardize
# Separating out the features
#x = filtered_df.drop(columns=col_to_delete).values
x = filtered_df.values

# x = RobustScaler().fit_transform(x)
x = StandardScaler().fit_transform(x)

pca = PCA(n_components=20)
pcs = pca.fit_transform(x)
pca_df = pd.DataFrame(data=pcs[:, :4], columns=["PC1", "PC2", "PC3", "PC4"])
pca_df = pd.concat([pca_df, my_df["condition"]], axis=1)
#pca_df = pca_df.dropna()
pca.explained_variance_ratio_

In [ ]:
fig = plt.figure(1, figsize=(8, 6))
ax = fig.add_subplot(111, projection="3d", elev=-150, azim=110)

X_reduced = PCA(n_components=3).fit_transform(x)
ax.scatter(
    X_reduced[:, 0],
    X_reduced[:, 1],
    X_reduced[:, 2],
    cmap=plt.cm.Set1,
    edgecolor="k",
)

ax.set_title("First three PCA directions")
ax.set_xlabel("1st eigenvector")
ax.set_ylabel("2nd eigenvector")
ax.set_zlabel("3rd eigenvector")

plt.show()

In [ ]:
plt.figure(figsize=(6, 4))
plt.plot(pca.explained_variance_ratio_, linewidth=2)

In [ ]:
pca_df

In [ ]:
sns.set_style("ticks")
sns.pairplot(pca_df, hue="condition", corner=True, palette=pal)

In [ ]:
sns_plot = sns.jointplot(data=pca_df, x="PC1", y="PC2", hue="condition", palette=pal)
plt.legend(bbox_to_anchor=(1.25, 1), loc='upper left')
plt.show()

In [ ]:
# Loadings for each principal component
loadings = pca.components_

# Convert loadings into a DataFrame for better readability
loadings_df = pd.DataFrame(
    loadings,
    columns=filtered_df.columns,
    index=[f"PC{i+1}" for i in range(loadings.shape[0])],
)

In [ ]:
# Sort the features by their contribution to PC1
correlated_pc1 = loadings_df.loc["PC1"].abs().sort_values(ascending=False)
print(correlated_pc1)

In [ ]:
loadings_df

In [ ]:
np.sum(pca.explained_variance_ratio_[:20])

In [ ]:
# get the index of the most important feature on EACH component
most_important_feats = [
    features[np.abs(pc).argmax()] for pc in pca.components_
]

# Build the dataframe
# Check the most contributing feature for each of the pcs
n_pcs = pca.components_.shape[0]
dic = {f"PC{i+1}": most_important_feats[i] for i in range(n_pcs)}
importance_df = pd.DataFrame(dic.items())
importance_df = importance_df.T
importance_df

In [ ]:
my_df.condition.value_counts()

In [ ]:
import umap
reducer = umap.UMAP(min_dist=1.0, random_state=2022)

embedding = reducer.fit_transform(x)
umap_df = pd.DataFrame(data=embedding, columns=["U1", "U2"])
umap_df = pd.concat([umap_df, my_df], axis=1)

plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=umap_df,
    x="U1",
    y="U2",
    hue="condition",
    palette=pal,
    s=150,
)

In [ ]:
plt.figure(figsize=(10, 6))
# sns.scatterplot(data=umap_df, x="U1", y="U2", hue="condition", s=150)
sns.kdeplot(
    data=umap_df,
    x="U1",
    y="U2",
    hue="condition",
    palette=pal,
    kind="kde",
    fill=True,
    levels=4,
    thresh=0.3,
    alpha=0.5,
    bw_adjust=0.75,
)

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=umap_df,
    x="U1",
    y="U2",
    hue="original_intensity_mean",
    s=150,
)

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=umap_df,
    x="U1",
    y="U2",
    hue="acquired_by",
    s=150,
)

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=umap_df,
    x="U1",
    y="U2",
    hue="year",
    s=150,
)

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=umap_df,
    x="U1",
    y="U2",
    hue="original_sigma_noise",
    s=150,
)

In [ ]:
from sklearn.cluster import KMeans

kmeans = KMeans(init="k-means++", n_clusters=10, random_state=0).fit(embedding)

umap_df["cluster"] = kmeans.labels_
plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=umap_df,
    x="U1",
    y="U2",
    hue="cluster",
    s=150,
)

In [ ]:
palette = {
    0: "grey",
    1: "grey",
    2: "grey",
    3: "grey",
    4: "red",
    5: "grey",
    6: "grey",
    7: "grey",
    8: "grey",
    9: "grey",
    10: "grey",
    11: "grey",
    12: "grey",
}

plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=umap_df,
    x="U1",
    y="U2",
    hue="cluster",
    palette=palette,
    s=150,
)

In [ ]:
filtered_df["cluster"] = kmeans.labels_
filtered_df = filtered_df[filtered_df["cluster"] != 4]

In [ ]:
#x = filtered_df.drop(columns=col_to_delete).values
x = filtered_df.values

# x = RobustScaler().fit_transform(x)
x = StandardScaler().fit_transform(x)

reducer = umap.UMAP(min_dist=1.0, random_state=2022)

embedding = reducer.fit_transform(x)
umap_df = pd.DataFrame(data=embedding, columns=["U1", "U2"])
umap_df = pd.concat([umap_df, my_df], axis=1)

plt.figure(figsize=(10, 6))
sns.scatterplot(data=umap_df, x="U1", y="U2", hue="condition", 
                palette=pal, s=150)

In [ ]:
plt.figure(figsize=(10, 6))
# sns.scatterplot(data=umap_df, x="U1", y="U2", hue="condition", s=150)
sns.kdeplot(
    data=umap_df,
    x="U1",
    y="U2",
    hue="condition",
    palette=pal,
    kind="kde",
    fill=True,
    levels=4,
    thresh=0.3,
    alpha=0.5,
    bw_adjust=0.75,
)

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=umap_df,
    x="U1",
    y="U2",
    hue="original_intensity_mean",
    s=150,
)

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=umap_df,
    x="U1",
    y="U2",
    hue="acquired_by",
    s=150,
)

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=umap_df,
    x="U1",
    y="U2",
    hue="year",
    s=150,
)

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=umap_df,
    x="U1",
    y="U2",
    hue="original_sigma_noise",
    s=150,
)

In [ ]:
filtered_df

In [ ]:
my_df.iloc[:, -36:]

In [ ]:
filtered_df = filtered_df.join(my_df.iloc[:, -36:])

In [ ]:
filtered_df.iloc[:, :-37]

### All features

In [ ]:
from joblib import parallel_backend
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import confusion_matrix, precision_recall_curve, roc_curve

import methods.ml_training as ml
import xgboost as xgb
import shap
shap.initjs()

seed = 2025
label_map = {'young': 1, 'aged': 0}
n_folds = 5
n_feats = 30

#ml_df = filtered_df.drop(columns=col_to_delete)
ml_df = filtered_df.iloc[:, :-37]

X = ml_df
y = [label_map[label] for label in filtered_df.condition]

kf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=seed)

In [ ]:
feat_sel_methods = ["none", "correlation", "RFE_LG", "RFE_RF", "RFE_XGB", 
                    "EMBED_LG", "EMBED_RF", "EMBED_XGB"]

benchmark_df = pd.DataFrame(columns=["model", "feat_selection", "accuracy", 
                                     "precision", "recall", "AUC", "F1"])

def do_benchmark(X, y, feat_sel_methods, kf, benchmark_df, n_feats, seed):

    for feat_sel in feat_sel_methods:

        print(f"-------- FEATURES: {feat_sel} --------\n")

        for fold, (train_idx, test_idx) in enumerate(kf.split(X, y), start=1):

            print(f"FOLD: {fold}\n")
    
            X_train, X_test, y_train, y_test = ml.prepare_train_test(X, y, train_idx, test_idx, feat_sel, n_feats=n_feats, seed=seed)
        
            print("Training Logistic Regression")
            benchmark_df = ml.fit_logreg(X_train, X_test, y_train, y_test, benchmark_df, feat_sel, fold, seed)
            print("Training Random Forest")
            benchmark_df = ml.optimize_fit_rf(X_train, X_test, y_train, y_test, benchmark_df, feat_sel, fold, seed)
            print("Training XGBoost")
            benchmark_df = ml.optimize_fit_xgb(X_train, X_test, y_train, y_test, benchmark_df, feat_sel, fold, seed)
            print("Training Calibrated XGBoost")
            benchmark_df = ml.optimize_fit_cal_xgb(X_train, X_test, y_train, y_test, benchmark_df, feat_sel, fold, seed)

    return benchmark_df


with parallel_backend('threading', n_jobs=16):
    benchmark_df = do_benchmark(X, y, feat_sel_methods, kf, benchmark_df, n_feats, seed)

In [ ]:
benchmark_df.to_csv("../results/pyradiomics/ml_benchmark_nowav_nobin_0.1_z_score_all_df.csv")

In [ ]:
benchmark_df = pd.read_csv(
    "../results/pyradiomics/ml_benchmark_nowav_nobin_0.1_z_score_all_df.csv",
    index_col=0,
)
benchmark_df["model"] = benchmark_df["model"].replace('LG', 'LR')
benchmark_df["feat_selection"] = benchmark_df["feat_selection"].replace('EMBED_LG', 'EMBED_LR')
benchmark_df["feat_selection"] = benchmark_df["feat_selection"].replace('RFE_LG', 'RFE_LR')

In [ ]:
n_folds=5

sum_df = benchmark_df.groupby(["fold"]).agg(
    accuracy_mean=("accuracy", "mean"),
    accuracy_std=("accuracy", "std"),
    precision_mean=("precision", "mean"),
    precision_std=("precision", "std"),
    recall_mean=("recall", "mean"),
    recall_std=("recall", "std"),
    AUC_mean=("AUC", "mean"),
    AUC_std=("AUC", "std"),
    F1_mean=("F1", "mean"),
    F1_std=("F1", "std")
).reset_index()

# Calculate the confidence interval margin for each metric
z_score = 1.96  # For a 95% confidence interval
for metric in ["accuracy", "precision", "recall", "AUC", "F1"]:
    ci = z_score * sum_df[f"{metric}_std"] / np.sqrt(n_folds)
    sum_df[f"{metric}_ci_lower"] = sum_df[f"{metric}_mean"] - ci
    sum_df[f"{metric}_ci_upper"] = sum_df[f"{metric}_mean"] + ci

sum_df

In [ ]:
sum_df = benchmark_df.groupby(["model", "feat_selection"]).agg(
    accuracy_mean=("accuracy", "mean"),
    accuracy_std=("accuracy", "std"),
    precision_mean=("precision", "mean"),
    precision_std=("precision", "std"),
    recall_mean=("recall", "mean"),
    recall_std=("recall", "std"),
    AUC_mean=("AUC", "mean"),
    AUC_std=("AUC", "std"),
    F1_mean=("F1", "mean"),
    F1_std=("F1", "std")
).reset_index()

# Calculate the confidence interval margin for each metric
z_score = 1.96  # For a 95% confidence interval
for metric in ["accuracy", "precision", "recall", "AUC", "F1"]:
    ci = z_score * sum_df[f"{metric}_std"] / np.sqrt(n_folds)
    sum_df[f"{metric}_ci_lower"] = sum_df[f"{metric}_mean"] - ci
    sum_df[f"{metric}_ci_upper"] = sum_df[f"{metric}_mean"] + ci

sum_df

In [ ]:
sum_df = benchmark_df.groupby(["feat_selection"]).agg(
    accuracy_mean=("accuracy", "mean"),
    accuracy_std=("accuracy", "std"),
    precision_mean=("precision", "mean"),
    precision_std=("precision", "std"),
    recall_mean=("recall", "mean"),
    recall_std=("recall", "std"),
    AUC_mean=("AUC", "mean"),
    AUC_std=("AUC", "std"),
    F1_mean=("F1", "mean"),
    F1_std=("F1", "std")
).reset_index()

# Calculate the confidence interval margin for each metric
z_score = 1.96  # For a 95% confidence interval
for metric in ["accuracy", "precision", "recall", "AUC", "F1"]:
    ci = z_score * sum_df[f"{metric}_std"] / np.sqrt(n_folds)
    sum_df[f"{metric}_ci_lower"] = sum_df[f"{metric}_mean"] - ci
    sum_df[f"{metric}_ci_upper"] = sum_df[f"{metric}_mean"] + ci

sum_df

### Benchmark comparison

In [ ]:
# Draw a nested barplot by species and sex
g = sns.catplot(
    data=benchmark_df,
    kind="bar",
    x="feat_selection",
    y="accuracy",
    hue="model",
    height=6,
    aspect=2,
)
g.set(ylim=(0.5, 0.85))

In [ ]:
# Draw a nested barplot by species and sex
g = sns.catplot(
    data=benchmark_df,
    kind="bar",
    x="feat_selection",
    y="F1",
    hue="model",
    height=6,
    aspect=2,
)
g.set(ylim=(0.5, 0.85))

In [ ]:
# Draw a nested barplot by species and sex
g = sns.catplot(
    data=benchmark_df,
    kind="bar",
    x="feat_selection",
    y="AUC",
    hue="model",
    height=6,
    aspect=2,
)
g.set(ylim=(0.5, 0.85))

In [ ]:
plt.figure(figsize=(4, 3))
model= "XGB"
feat_sel = "EMBED_RF"

benchmark_df = pd.DataFrame(columns=["model", "feat_selection", "accuracy", 
                                     "precision", "recall", "AUC", "F1"])

prob_df = pd.DataFrame(columns=["prob", "pred"], index=filtered_df.index)
aucs_df = pd.DataFrame(columns=["TPR", "FPR", "n_feats"])

n_feats_list = [10, 20, 30, 40, 50]

with parallel_backend('threading', n_jobs=24):

    for n_feats in n_feats_list:
        
        for fold, (train_idx, test_idx) in enumerate(kf.split(X, y), start=1):
    
            X_train, X_test, y_train, y_test = ml.prepare_train_test(
                X, y, train_idx, test_idx, 
                feat_sel=feat_sel, n_feats=n_feats, seed=seed
            )

            print(X_train.shape)
            
            le = LabelEncoder().fit(y_train)
            y_train_xgb = le.transform(y_train)
            y_test_xgb = le.transform(y_test)
        
            best_clf = ml.gridsearch_XGB(X_train, y_train_xgb, seed)
            
            clf = xgb.XGBClassifier(
                tree_method="hist",
                n_estimators=best_clf.best_estimator_.n_estimators,
                max_depth=best_clf.best_estimator_.max_depth,
            ).fit(X_train, y_train_xgb)

            y_pred = clf.predict(X_test)
            y_prob = clf.predict_proba(X_test)[:, 1]

            # Compute ROC curve
            fpr, tpr, _ = roc_curve(y_test, y_prob)
    
            these_idx = prob_df.iloc[list(test_idx)].index
            prob_df.loc[these_idx, "pred"] = y_pred
            prob_df.loc[these_idx, "prob"] = y_prob
            
            ml.update_benchmark_metrics(y_test, y_pred, feat_sel, model, fold, benchmark_df)

        aucs_df = pd.concat([aucs_df, pd.DataFrame({"TPR":tpr, "FPR":fpr, "n_feats":n_feats})])

In [ ]:
plt.figure(figsize=(4, 3))
sns.lineplot(data=aucs_df, x="FPR", y="TPR", hue="n_feats")

In [ ]:
def plot_umap(X):
    
    col_names = X.columns
    scaler = StandardScaler()
    X_std = scaler.fit_transform(X)
    X_std = pd.DataFrame(X_std, columns=col_names)
    
    reducer = umap.UMAP(min_dist=1.0, random_state=2022)
    
    embedding = reducer.fit_transform(X_std)
    umap_df = pd.DataFrame(data=embedding, columns=["U1", "U2"])
    umap_df = pd.concat([umap_df, filtered_df, prob_df], axis=1)
    
    plt.figure(figsize=(10, 6))
    sns.scatterplot(data=umap_df, x="U1", y="U2", hue="condition", s=150, palette=pal)
    plt.show()

    sns.jointplot(
        data=umap_df,
        x="U1",
        y="U2",
        hue="condition",
        kind="kde",
        fill=True,
        levels=4,
        thresh=0.4,
        alpha=0.5,
        bw_adjust=0.9,
        palette=pal,
    )
    plt.show()

In [ ]:
model= "XGB"
feat_sel = "EMBED_RF"
n_feats = 30

benchmark_df = pd.DataFrame(columns=["model", "feat_selection", "accuracy", 
                                     "precision", "recall", "AUC", "F1"])

prob_df = pd.DataFrame(columns=["prob", "pred"], index=filtered_df.index)
aucs_df = pd.DataFrame(columns=["TPR", "FPR", "fold"])
pr_df = pd.DataFrame(columns=["Precision", "Recall", "fold"])
conf_df = pd.DataFrame(columns=["Confusion", "index", "fold"])

shap_df = pd.DataFrame()

with parallel_backend('threading', n_jobs=16):
    
    for fold, (train_idx, test_idx) in enumerate(kf.split(X, y), start=1):

        X_train, X_test, y_train, y_test = ml.prepare_train_test(
            X, y, train_idx, test_idx, feat_sel, n_feats=n_feats, seed=seed
        )
        
        print(X_train.shape)
        
        le = LabelEncoder().fit(y_train)
        y_train_xgb = le.transform(y_train)
        y_test_xgb = le.transform(y_test)
    
        best_clf = ml.gridsearch_XGB(X_train, y_train_xgb, seed)
        
        clf = xgb.XGBClassifier(
            tree_method="hist",
            n_estimators=best_clf.best_estimator_.n_estimators,
            max_depth=best_clf.best_estimator_.max_depth,
        ).fit(X_train, y_train_xgb)
        
        y_pred = clf.predict(X_test)
        y_prob = clf.predict_proba(X_test)[:, 1]

        these_idx = prob_df.iloc[list(test_idx)].index
        prob_df.loc[these_idx, "pred"] = y_pred
        prob_df.loc[these_idx, "prob"] = y_prob
        
        df = ml.update_benchmark_metrics(y_test, y_pred, feat_sel, model, fold, benchmark_df)
        acc = df["accuracy"][0]
        f1 = df["F1"][0]
        roc_auc = df["AUC"][0]
        
        # explain all the predictions in the test set
        # X_train_subset = shap.sample(X_train, 100)
        explainer = shap.TreeExplainer(clf)
        explanation = explainer(X_test)
        shap_values = explanation.values
        these_shaps = pd.DataFrame(shap_values, columns=X_test.columns)
        shap_df = pd.concat([shap_df, these_shaps])

        mean_shap_values = these_shaps.abs().mean(axis=0)
        top = mean_shap_values.sort_values(ascending=False).head(15)
        top_names = top.index.to_list()
        plot_umap(X[top_names])

        shap.plots.beeswarm(explanation, max_display=20, show=False)
        plt.title(f"Fold {fold}  |  Acc {acc:.3f}  |  AUC {roc_auc:.3f}  |  F1 {f1:.3f}", fontsize=20)
        plt.show()
        
        # Compute ROC curve
        fpr, tpr, _ = roc_curve(y_test, y_prob)
        aucs_df = pd.concat([aucs_df, pd.DataFrame(
            {"TPR":tpr, "FPR":fpr, "fold":fold})]
        )

        # Compute PR curve
        precision, recall, _ = precision_recall_curve(y_test, y_prob)
        pr_df = pd.concat([pr_df, pd.DataFrame(
            {"Precision":precision, "Recall":recall, "fold":fold})]
        )

        # Compute confusion matrix
        conf_mat = confusion_matrix(y_test, y_pred, normalize="true").flatten()
        conf_df = pd.concat([conf_df, pd.DataFrame(
            {"Confusion":conf_mat, "index":[1, 2, 3, 4], "fold":fold})]
        )


In [ ]:
shap_df.to_csv("../results/pyradiomics/supp_table3_SHAP_df.csv")

In [ ]:
def plot_conf_mat(conf_df):
    
    summ_df = conf_df.groupby("index")["Confusion"].agg(["mean", "std"]).reset_index()
    conf_plot = summ_df["mean"].to_numpy().reshape((2, 2))
    
    annot = np.array([
        f"{m:.3f}% \n ± {s:.3f}" for m, s in zip(summ_df["mean"], summ_df["std"])
    ]).reshape((2, 2))
    
    plt.figure(figsize=(4, 3))
    sns.heatmap(conf_plot, annot=annot, fmt='', cmap="Blues")
    plt.xticks([0.5, 1.5], ["Aged", "Young"])
    plt.yticks([0.5, 1.5], ["Aged", "Young"])
    plt.show()

plot_conf_mat(conf_df)

In [ ]:
plt.figure(figsize=(4, 3))
sns.lineplot(data=aucs_df, x="FPR", y="TPR", hue="fold")
plt.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Random (AUC=0.5)")
plt.show()

In [ ]:
plt.figure(figsize=(4, 3))
sns.lineplot(data=pr_df, x="Recall", y="Precision", hue="fold")
plt.ylim((0.4, 1))
baseline = sum(y) / len(y)
plt.axhline(y=baseline, color="gray", linestyle="--", label="Random Baseline")
plt.show()

In [ ]:
pairs = [("young", "aged")]

palette = {
    "young": (0.00392, 0.45098, 0.69803),
    "aged": (0.00784, 0.61960, 0.45098),
    "aged_treated_RhoAi": (0.8, 0.47058, 0.73725),
    "8um": (0.79215, 0.56862, 0.38039),
    "5um": (0.87058, 0.56078, 0.01960),
    "3um": (0.83529, 0.36862, 0.0),
}

for feat in top.index.to_list():
    print(feat.index)
    plt.figure(figsize=(4, 4))
    b = sns.boxplot(
        data=my_df, x="condition", y=feat, hue="condition", palette=pal
    )
    print("Aged: ", my_df[my_df["condition"] == "aged"][feat].mean())
    print("Young: ", my_df[my_df["condition"] == "young"][feat].mean())
    
    # If seaborn keeps plotting in a separate axis, use plt.gca() to get current axis
    annotator = Annotator(ax=b, data=my_df, x="condition", y=feat, pairs=pairs)
    annotator.configure(
        test="Mann-Whitney", text_format="simple", loc="outside", comparisons_correction="Bonferroni"
    )
    annotator.apply_and_annotate()

    fontsize = 12
    if len(feat) > 42:
        fontsize=10
    b.set_ylabel(feat, fontsize=fontsize)
    plt.show()

In [ ]:
# Take only the SHAP values that appear in all folds
mean_shap_values = shap_df.dropna(axis=1).abs().mean(axis=0)
top = mean_shap_values.sort_values(ascending=False)
top_names = top.index.to_list()
print(len(top))
top_names

In [ ]:
col_names = X.columns
scaler = StandardScaler()
X_std = scaler.fit_transform(X)
X_std = pd.DataFrame(X_std, columns=col_names)

reducer = umap.UMAP(min_dist=1.0, random_state=2022)

embedding = reducer.fit_transform(X_std[top_names])

umap_df = pd.DataFrame(data=embedding, columns=["U1", "U2"])
umap_df = pd.concat([umap_df, filtered_df, prob_df], axis=1)

plt.figure(figsize=(10, 6))
sns.scatterplot(data=umap_df, x="U1", y="U2", hue="condition", s=150, palette=pal)

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(data=umap_df, x="U1", y="U2", hue="prob", s=150)

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(data=umap_df, x="U1", y="U2", hue="pred", s=150)

In [ ]:
plt.figure(figsize=(10, 6))
# sns.scatterplot(data=umap_df, x="U1", y="U2", hue="condition", s=150)
sns.jointplot(
    data=umap_df,
    x="U1",
    y="U2",
    hue="condition",
    kind="kde",
    fill=True,
    levels=4,
    thresh=0.4,
    alpha=0.5,
    bw_adjust=0.9,
    palette=pal,
)
plt.ylim(5, 18)
plt.xlim(-4, 13)